# Docker Host Setup for Jupyter Notebooks

This notebook demonstrates how to create a basic Docker host for running Jupyter instances in your WSL Ubuntu environment. We'll set up a containerized Jupyter environment that's portable, scalable, and easy to manage.

## Benefits of Docker for Jupyter
- **Isolation**: Clean environment for each project
- **Portability**: Same environment across different machines
- **Reproducibility**: Consistent setup for all team members
- **Scalability**: Easy to spin up multiple instances
- **Security**: Contained environment with controlled access

## 1. Install Docker and Docker Compose

First, we need to install Docker Engine and Docker Compose on our WSL Ubuntu environment.

In [ ]:
# Install Docker Engine
!sudo apt update
!sudo apt install -y apt-transport-https ca-certificates curl gnupg lsb-release

# Add Docker's official GPG key
!curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg

# Set up the stable repository
!echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null

# Install Docker Engine
!sudo apt update
!sudo apt install -y docker-ce docker-ce-cli containerd.io docker-compose-plugin

In [ ]:
# Add user to docker group and start Docker service
!sudo usermod -aG docker $USER
!sudo service docker start

# Verify Docker installation
!docker --version
!docker compose version

# Test Docker with hello-world
!docker run hello-world

## 2. Create Dockerfile for Jupyter

Now we'll create a Dockerfile that sets up a Jupyter environment with necessary dependencies and configurations.

In [ ]:
# Create Dockerfile for our Jupyter setup
dockerfile_content = """FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    git \\
    curl \\
    vim \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Install Python packages
RUN pip install --no-cache-dir \\
    jupyter \\
    jupyterlab \\
    numpy \\
    pandas \\
    matplotlib \\
    seaborn \\
    plotly \\
    scikit-learn \\
    requests

# Create a non-root user
RUN useradd -m -s /bin/bash jupyter

# Create directories
RUN mkdir -p /home/jupyter/notebooks && \\
    chown -R jupyter:jupyter /home/jupyter

# Set up Jupyter configuration
USER jupyter
WORKDIR /home/jupyter

# Generate Jupyter config
RUN jupyter notebook --generate-config && \\
    echo "c.NotebookApp.ip = '0.0.0.0'" >> ~/.jupyter/jupyter_notebook_config.py && \\
    echo "c.NotebookApp.port = 8888" >> ~/.jupyter/jupyter_notebook_config.py && \\
    echo "c.NotebookApp.open_browser = False" >> ~/.jupyter/jupyter_notebook_config.py && \\
    echo "c.NotebookApp.allow_root = False" >> ~/.jupyter/jupyter_notebook_config.py

# Expose port
EXPOSE 8888

# Start Jupyter
CMD ["jupyter", "lab", "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root"]
"""

# Write Dockerfile
with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)
    
print("Dockerfile created successfully!")

## 3. Set Up Docker Compose Configuration

Create a docker-compose.yml file to define the Jupyter service with port mapping, environment variables, and volume mounting.

In [ ]:
# Create docker-compose.yml file
compose_content = """version: '3.8'

services:
  jupyter:
    build: .
    container_name: jupyter-lab
    ports:
      - "8888:8888"
    volumes:
      - ./notebooks:/home/jupyter/notebooks
      - ./data:/home/jupyter/data
    environment:
      - JUPYTER_ENABLE_LAB=yes
      - JUPYTER_TOKEN=your-secure-token-here
    restart: unless-stopped
    networks:
      - jupyter-network

networks:
  jupyter-network:
    driver: bridge

volumes:
  notebooks-data:
    driver: local
"""

# Write docker-compose.yml
with open('docker-compose.yml', 'w') as f:
    f.write(compose_content)
    
print("docker-compose.yml created successfully!")

## 4. Build and Run Jupyter Container

Now we'll build the Docker image and run the Jupyter container using Docker Compose commands.

In [ ]:
# Create necessary directories
!mkdir -p notebooks data

# Build and start the Jupyter container
!docker compose build --no-cache
print("Docker image built successfully!")

# Start the container in detached mode
!docker compose up -d
print("Jupyter container started!")

# Check container status
!docker compose ps

## 5. Configure Volume Mounting

Volume mounting ensures that your notebooks and data persist between container restarts. Let's set up the directory structure and verify the mounting.

In [ ]:
# Create a sample notebook to test volume mounting
sample_notebook = """{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Welcome to Dockerized Jupyter!\\n",
    "\\n",
    "This notebook is running inside a Docker container and is mounted to your local filesystem."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\\n",
    "import pandas as pd\\n",
    "import numpy as np\\n",
    "import matplotlib.pyplot as plt\\n",
    "\\n",
    "print(f\\"Python version: {sys.version}\\")\\n",
    "print(f\\"Pandas version: {pd.__version__}\\")\\n",
    "print(f\\"NumPy version: {np.__version__}\\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}"""

# Write sample notebook
with open('notebooks/welcome.ipynb', 'w') as f:
    f.write(sample_notebook)

print("Sample notebook created in ./notebooks/welcome.ipynb")

# Verify volume mounting by checking container file system
!docker compose exec jupyter ls -la /home/jupyter/notebooks

## 6. Access and Test Jupyter Instance

Now let's access the Jupyter instance through the browser and verify that everything is working correctly.

In [ ]:
# Get the Jupyter container logs to find the access URL
!docker compose logs jupyter

print("\\n" + "="*50)
print("ACCESS INSTRUCTIONS:")
print("="*50)
print("1. Open your web browser")
print("2. Navigate to: http://localhost:8888")
print("3. Use token: 'your-secure-token-here' (or check logs above)")
print("4. You should see JupyterLab interface")
print("5. Open the 'welcome.ipynb' notebook to test")

# Check container health
!docker compose ps

print("\\n" + "="*50)
print("CONTAINER MANAGEMENT COMMANDS:")
print("="*50)
print("Stop container:    docker compose down")
print("Start container:   docker compose up -d")
print("View logs:         docker compose logs jupyter")
print("Enter container:   docker compose exec jupyter bash")
print("Rebuild:           docker compose build --no-cache")

## Conclusion and Next Steps

🎉 **Congratulations!** You now have a fully functional Docker-based Jupyter environment running in your WSL Ubuntu setup.

### What we've accomplished:
- ✅ Installed Docker and Docker Compose
- ✅ Created a custom Dockerfile for Jupyter
- ✅ Set up Docker Compose configuration
- ✅ Configured volume mounting for persistence
- ✅ Created a containerized Jupyter environment

### Next Steps:
1. **Customize your environment**: Add more Python packages to the Dockerfile
2. **Security**: Change the default token in docker-compose.yml
3. **Multiple environments**: Create different containers for different projects
4. **Backup**: Set up automated backups of your notebooks directory
5. **Scaling**: Use Docker Swarm or Kubernetes for production deployments

### Useful Commands for Daily Use:
```bash
# Start Jupyter
docker compose up -d

# Stop Jupyter
docker compose down

# View logs
docker compose logs -f jupyter

# Update and rebuild
docker compose build --no-cache && docker compose up -d
```

Your Jupyter notebooks are now portable, isolated, and easy to manage! 🐳📊